In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim

In [2]:
import torchvision
from torchvision import transforms, datasets

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.Resize((32,32))
])

In [4]:
from torch.utils.data import random_split

In [8]:
data = datasets.ImageFolder(root=path, transform=transform)

train_size = int(0.8 * len(data))
test_size = int(len(data) - train_size)

In [6]:
import kagglehub

In [7]:
import kagglehub
path = kagglehub.dataset_download("tongpython/cat-and-dog")

Using Colab cache for faster access to the 'cat-and-dog' dataset.


In [9]:
torch.manual_seed(42)

train_dataset, test_dataset = random_split(data, [train_size, test_size], torch.Generator().manual_seed(42))

In [10]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size = 64, shuffle = True)

In [17]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.AdaptiveAvgPool2d((7, 7))
        )

        self.fc_layers = nn.Sequential(

            nn.Linear(6272, 256),
            nn.ReLU(),
            nn.Dropout(0.5), # Recommended to prevent overfitting
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)

        return x


In [18]:
model = CNN()

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())
#

In [20]:
epochs = 10

for i in range(epochs):
  model.train()

  epoch_running_loss = 0.0

  for images, labels in train_loader:
    optimizer.zero_grad()
    outputs = model.forward(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    epoch_running_loss += loss.item()
  print(f"for epoch {i+1}, loss is {epoch_running_loss}")

for epoch 1, loss is 70.40249851346016
for epoch 2, loss is 64.55029121041298
for epoch 3, loss is 64.2999732196331
for epoch 4, loss is 64.14201444387436
for epoch 5, loss is 63.799393981695175
for epoch 6, loss is 63.78978955745697
for epoch 7, loss is 63.71060860157013
for epoch 8, loss is 63.735994935035706
for epoch 9, loss is 63.76284816861153
for epoch 10, loss is 63.6881802380085


In [23]:
correct = 0
total = 0

with torch.no_grad():
  for images, labels in test_loader:
    outputs = model.forward(images)
    _, predicted = torch.max(outputs, 1)

    correct += (predicted == labels).sum().item()
    total += labels.size(0)

print("accuracy = {}".format(correct/ total *100))

accuracy = 79.31206380857428


In [27]:
from sklearn.metrics import precision_score, accuracy_score,recall_score, confusion_matrix, f1_score, r2_score

In [26]:
print(confusion_matrix(labels, predicted))

[[ 0  4]
 [ 0 18]]


In [28]:
print(f"Accuracy score {accuracy_score(labels, predicted)}\nPrecision Score{precision_score(labels, predicted)}\nRecall Score {recall_score(labels, predicted)}\nf1 Score {f1_score(labels, predicted)}\nr2 Score {r2_score(labels, predicted)}")

Accuracy score 0.8181818181818182
Precision Score0.8181818181818182
Recall Score 1.0
f1 Score 0.9
r2 Score -0.22222222222222232
